# ATU: Reinforcement Learning Intraday Trading Pipeline on Google Colab GPU

This notebook configures and runs the **ATU (Automated Trading Unit)** reinforcement learning intraday trading system on Google Colab with GPU acceleration.

### Architecture & Pipeline Overview:
- **Primary Development Environment:** Google Antigravity IDE (Windows)
- **Version Control & Synchronization:** GitHub (`https://github.com/SUZUB/ATU.git`)
- **GPU Compute Engine:** Google Colab (NVIDIA T4 / V100 / A100)
- **Persistent Storage:** Google Drive (`/content/drive/MyDrive/ATU_Models/`)

### Sequential 4-File Data Flow:
1. **File 1 (`market_data_processor.py`):** Fetches/generates 5-min OHLCV bars + normalized technical indicators (RSI, MACD, ATR, Bollinger Bands, ADX).
2. **File 2 (`sentiment_processor.py`):** Runs `ProsusAI/finbert` on CUDA GPU to extract continuous financial sentiment [-1.0, 1.0].
3. **State Assembly:** Merges features into `final_state_vector` and chronologically splits into **80% Train** and **20% Test**.
4. **File 3 (`constrained_trading_env.py`):** Gymnasium environment enforcing 1% dynamic risk, 2% target, 3 trades/day, round-trip fees, 15:15 MIS auto-square-off, and unbroken continuous compounding.
5. **File 4 (`train_and_backtest.py`):** Vectorizes environment with `DummyVecEnv`, trains Stable-Baselines3 `PPO` on GPU, saves model checkpoint to Google Drive, and executes out-of-sample continuous backtest.

## 1. Environment & Python Version Checks
Verify the Python runtime and system environment.

In [ ]:
import sys
import platform
import os

print("=" * 60)
print("ENVIRONMENT & RUNTIME CHECK")
print("=" * 60)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"Python Executable:{sys.executable}")
print(f"Operating System: {platform.system()} {platform.release()}")
print(f"Current Working Dir: {os.getcwd()}")
print("=" * 60)

## 2. GPU Detection & Hardware Acceleration
Check for active NVIDIA GPU (such as T4) and verify CUDA availability for PyTorch and Transformers.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print("=" * 60)
print("GPU DETECTION")
print("=" * 60)
print(f"CUDA Available:     {cuda_available}")

if cuda_available:
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    vram_gb = torch.cuda.get_device_properties(current_device).total_memory / (1024**3)
    print(f"Device Index:       {current_device}")
    print(f"Device Name:        {device_name}")
    print(f"Device Count:       {device_count}")
    print(f"Total VRAM:         {vram_gb:.2f} GB")
    print("Status: GPU acceleration enabled for PPO training & FinBERT inference!")
else:
    print("Status: Running on CPU.")
    print("To enable GPU in Colab: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
print("=" * 60)

!nvidia-smi

## 3. Google Drive Integration (Persistent Model Checkpoints)
Mount Google Drive to persist trained model checkpoints, logs, and outputs across Colab runtime disconnects.

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define persistent checkpoint directory
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/ATU_Models')
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent checkpoint directory ready: {DRIVE_MODEL_DIR}")
!ls -ld /content/drive/MyDrive/ATU_Models

## 4. GitHub Repository Cloning & Synchronization
Clone the repository or pull the latest changes pushed from your Antigravity local workspace.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/SUZUB/ATU.git"
PROJECT_DIR = Path("/content/ATU")

if not PROJECT_DIR.exists():
    print(f"Cloning {REPO_URL} into {PROJECT_DIR}...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print(f"{PROJECT_DIR} already exists. Pulling latest updates from GitHub...")
    %cd {PROJECT_DIR}
    !git pull origin main

%cd {PROJECT_DIR}
!git status

## 5. Dependency Installation
Install all required packages from `requirements.txt`.

In [ ]:
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("All dependencies successfully installed!")

## 6. Project Initialization & Component Verification
Verify all project modules and entry points are loadable.

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Verify all core components
from market_data_processor import MarketDataProcessor
from sentiment_processor import SentimentProcessor
from constrained_trading_env import ConstrainedTradingEnv
from src.data_processing import (
    create_final_state_vector,
    generate_intraday_5min_market_data,
    load_archive_headlines,
)
from src.model import create_vectorized_env, create_ppo_agent
from train_and_backtest import run_pipeline

print("Component Verification Complete:")
print("  - File 1 (MarketDataProcessor):     [OK]")
print("  - File 2 (SentimentProcessor):      [OK]")
print("  - File 3 (ConstrainedTradingEnv):    [OK]")
print("  - File 4 (Main Execution Pipeline):  [OK]")

## 7. Dataset Loading & Inspection
Inspect headlines and market data for state vector construction.

In [ ]:
# Load archive headlines if available; otherwise use realistic fallback
headlines_sample = load_archive_headlines(archive_dir=PROJECT_DIR / "archive", ticker="AAPL", max_rows=50)
print(f"Headlines sample ({len(headlines_sample)} rows):")
display(headlines_sample.head(5))

# Test market data generator
market_sample = generate_intraday_5min_market_data(n_rows=200, initial_price=150.0)
print(f"Market 5-min candles ({len(market_sample)} rows):")
display(market_sample.head(5))

## 8. Training Execution (Using Project Entry Point)
Execute the main training pipeline using `train_and_backtest.py`.
Model checkpoints are saved directly to Google Drive (`/content/drive/MyDrive/ATU_Models/ppo_trading_agent_v1`).

In [ ]:
compute_device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = str(DRIVE_MODEL_DIR / "ppo_trading_agent_v1")

print(f"Target Compute Device:    {compute_device}")
print(f"Google Drive Checkpoint:  {checkpoint_path}")

# Execute main entry point with 200,000 steps
!python train_and_backtest.py \
    --timesteps 200000 \
    --ticker AAPL \
    --device {compute_device} \
    --model-save-path "{checkpoint_path}"

## 9. Interactive Programmatic Pipeline & Performance Metrics
Alternatively, run or evaluate the pipeline programmatically in Python to inspect equity metrics and Sortino ratio directly.

In [ ]:
from train_and_backtest import run_pipeline

# Run a fast verification or evaluation
model, final_balance, net_return, sortino = run_pipeline(
    total_timesteps=10000,
    ticker="AAPL",
    use_live_market_data=False,
    use_finbert=True if torch.cuda.is_available() else False,
    model_save_path=checkpoint_path,
    device=compute_device,
)

print(f"=== EVALUATION SUMMARY ===")
print(f"Final Balance: ₹{final_balance:,.2f}")
print(f"Net Return:    {net_return:+.2f}%")
print(f"Sortino Ratio: {sortino:.4f}")

## 10. Model Checkpoint Verification in Google Drive
Confirm that model weights are safely written to persistent Google Drive storage.

In [ ]:
target_zip = Path(f"{checkpoint_path}.zip")

if target_zip.exists():
    size_kb = target_zip.stat().st_size / 1024
    print(f"CONFIRMED: Checkpoint successfully persisted in Google Drive!")
    print(f"Path: {target_zip}")
    print(f"Size: {size_kb:.2f} KB")
else:
    print(f"Notice: Checkpoint {target_zip} not detected yet.")

!ls -lh /content/drive/MyDrive/ATU_Models/

## 11. Download Results to Local Machine
Download the trained model checkpoint to your local machine if needed.

In [ ]:
from google.colab import files

def download_model():
    if target_zip.exists():
        print(f"Downloading {target_zip.name} to your local computer...")
        files.download(str(target_zip))
    else:
        print(f"File {target_zip} does not exist.")

# Run the function below to trigger browser download:
# download_model()